# 4. Save Permutations for Session Drift

This notebook generates permutation-based null distributions for the session-drift analyses.

It uses the combined ROI outputs from:

`3. Combine_Session__ROIs.ipynb`

and creates permuted versions of the session order. These permutations are later used to test whether observed drift across sessions is stronger than expected by chance.

# Notebook Position in the Pipeline

Previous notebook:

`3. Combine_Session__ROIs.ipynb`

Current notebook:

`4.Save_Permutations_for_Session_Drift.ipynb`

Next notebooke use the output for:

`5. sim_pop_response_expand.ipynb`

# Inputs and Outputs

## Input

This notebook loads combined ROI files:

`regressSessCombineROI_sub{subject}{normalization}.pkl`

Example:

`regressSessCombineROI_sub1_equalStd.pkl`

These files contain session-wise regression outputs after ROI subdivisions were combined.

## Output

This notebook saves permutation files:

`perm_roi{roi}{nperms}{normalization}.pkl`

Example:

`perm_roi31000_zeroMean.pkl`

The saved file contains the real session-distance metrics and the corresponding permuted null distributions.

# Important Note for Adapting This Notebook to a New Dataset

This notebook was originally written for the NSD dataset and assumes the output structure created by the previous notebooks in this pipeline.

If you adapt this notebook to another dataset, you should not change the logic of the permutation test first.  
Instead, first make sure your new dataset produces the same required inputs.

The main dataset-specific parts are:

1. Number of subjects
2. Number of sessions per subject
3. ROI names and ROI indexing
4. File paths and file naming convention
5. Shape of the saved regression outputs
6. Whether a mean split was appended in the previous notebook
7. Which metric should be used for selecting good voxels

# Run code

In [ ]:
import os
import numpy as np
import pickle
import time
from typing import List, Dict, Any
from google.colab import drive

# -------------------------
# Mount Drive
# -------------------------
drive.mount('/content/drive', force_remount=True)

# -------------------------
# Paths
# -------------------------
SAVE_DIR = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
os.makedirs(SAVE_DIR, exist_ok=True)

# -------------------------
# Helpers
# -------------------------

def get_zscore_suffix(to_zscore: int) -> str:
    if to_zscore == 1:
        return "_zscore"
    elif to_zscore == 2:
        return "_zeroMean"
    elif to_zscore == 3:
        return "_equalStd"
    elif to_zscore == 4:
        return "_zeroROImean"
    return ""

# ============================================================
# HELPER FUNCTION: AUTOCORRELATION
# ============================================================
# Computes autocorrelation across session order for one time series.
#
# Used to estimate temporal stability/drift in:
#     - mean beta responses
#     - beta standard deviations
#     - constant coefficients
#     - coefficient magnitudes
#
# The same function is also applied after permuting session order.
# ============================================================
def autocorr_1d(x: np.ndarray, max_lag: int) -> np.ndarray:
    """
    Approximate MATLAB's autocorr(x) up to max_lag.
    Returns vector of length max_lag+1 (lags 0..max_lag).
    """
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 1 or x.size == 0:
        return np.full(max_lag + 1, np.nan, dtype=np.float32)

    x = x - np.nanmean(x)
    n = x.size
    var = np.nanvar(x)
    if var == 0 or np.isnan(var):
        return np.full(max_lag + 1, np.nan, dtype=np.float32)

    # full autocorrelation using FFT would be faster, but np.correlate is fine here
    result = np.correlate(x, x, mode="full")  # length 2n-1
    mid = result.size // 2
    ac = result[mid:mid + max_lag + 1] / (var * n)
    return ac.astype(np.float32)


# ============================================================
# MAIN FUNCTION: SAVE SESSION-ORDER PERMUTATIONS
# ============================================================
# This function loads combined ROI regression outputs and computes:
#
#     1. Observed session-distance metrics in the real order
#     2. Permuted session-distance metrics after shuffling session order
#     3. Autocorrelation measures in real and permuted orders
#
# The resulting permutation file is used later for statistical testing:
#
#     observed drift vs. null drift expected by random session order
# ============================================================
def save_perms_expand(
    rois: List[int] = None,
    to_zscore: int = 0,
    nperms: int = 1000,
    r2thresh: float = 0.0,
    fixedFirst: bool = False,
    save_dir: str = SAVE_DIR,
    subjects: List[int] = None,
) -> str:
    """
    Python/Colab version of savePerms_expand.m

    Args:
        rois       : list of visual-region indices (1..4). Example: [1] for V1 only.
        to_zscore  : normalization flag used in previous steps (0..4).
        nperms     : number of permutations for session order.
        r2thresh   : pRF R² threshold; voxels with r2 <= r2thresh are discarded.
        fixedFirst : if True, keep session 1 fixed and permute sessions 2..30.
        save_dir   : folder where regressSessCombineROI_sub*.pkl live.

    Returns:
        Path to the saved .pkl file with permutation results.
    """
    t0 = time.time()

    if rois is None:
        rois = [1]   # default: V1 only (as in usage example savePerms_expand(1,...))

    save_dir = os.path.abspath(save_dir)
    zscore_str = get_zscore_suffix(to_zscore)

    print(f"\n=== save_perms_expand(rois={rois}, to_zscore={to_zscore}, nperms={nperms}, r2thresh={r2thresh}, fixedFirst={fixedFirst}) ===")
    print("Using save_dir:", save_dir)

    fixedFirstStr = "_fixedFirst_" if fixedFirst else ""

    # settings
    toNormalize = 0  # as in MATLAB (they set this but keep it 0)
    useMedian = 1    # 1 = median over voxels

    # subjects: which subject IDs to use (e.g. [3] or [1,2,3])
    if subjects is None:
        subjects = [1]  # default, like MATLAB [1:8] but reduced
    nsubjects = len(subjects)

    # In the original code, they hard-coded nsessions=30 (max used).
    nsessions = 30
    minSessions = nsessions  # same name as MATLAB

    nrois = 4  # max number of visual regions; we will only fill selected rois

    # ----- r2thresh string -----
    if r2thresh > 0:
        r2threshStr = f"r2thresh{r2thresh:4.0f}"
    else:
        r2threshStr = ""

    # ----- Distance & lag matrices (30 × 30) -----
    idx = np.arange(nsessions)
    distMatrix = np.abs(idx[:, None] - idx[None, :])  # toeplitz(0:nsessions-1)
    # distMatrix stores absolute distance between sessions.
    lagMatrix = idx[None, :] - idx[:, None]           # toeplitz(0:-1:1-nsessions,0:nsessions-1)
    # lagMatrix stores signed session lag and is used to average both directions of the session-pair matrix.

    # ----- Prealloc global arrays -----
    # mean over voxels (third dim = sessions)
    meanBetas = np.full((nrois, nsubjects, nsessions), np.nan, dtype=np.float32)
    meanStdBetas = np.full((nrois, nsubjects, nsessions), np.nan, dtype=np.float32)
    meanConstantCoef = np.full((nrois, nsubjects, nsessions), np.nan, dtype=np.float32)
    meanConstantOriCoef = np.full((nrois, nsubjects, nsessions), np.nan, dtype=np.float32)

    # subSessions: which sessions exist per subject (1..nsessions)
    subSessions = np.zeros((nsubjects, nsessions), dtype=np.int8)

    # Distance-based summary arrays (lag 1..nsessions-1)
    rssSplit = [np.full((nsubjects, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    rssOriSplit = [np.full((nsubjects, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    r2split = [np.full((nsubjects, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    r2oriSplit = [np.full((nsubjects, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    pearsonR = [np.full((nsubjects, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    pearsonRori = [np.full((nsubjects, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]

    rssDist = [np.full((nsubjects, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    rssOriDist = [np.full((nsubjects, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    r2Dist = [np.full((nsubjects, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    r2OriDist = [np.full((nsubjects, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    pearsonDist = [np.full((nsubjects, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    pearsonOriDist = [np.full((nsubjects, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]

    # Distances per permutation
    rssDistPerm = [np.full((nsubjects, nperms, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    rssOriDistPerm = [np.full((nsubjects, nperms, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    r2DistPerm = [np.full((nsubjects, nperms, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    r2OriDistPerm = [np.full((nsubjects, nperms, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    pearsonDistPerm = [np.full((nsubjects, nperms, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]
    pearsonOriDistPerm = [np.full((nsubjects, nperms, nsessions - 1), np.nan, dtype=np.float32) for _ in range(nrois)]

    # Full split×split matrices per permutation
    rssPerm = [np.full((nsubjects, nperms, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    rssOriPerm = [np.full((nsubjects, nperms, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    r2perm = [np.full((nsubjects, nperms, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    r2oriPerm = [np.full((nsubjects, nperms, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    pearsonPerm = [np.full((nsubjects, nperms, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]
    pearsonOriPerm = [np.full((nsubjects, nperms, nsessions, nsessions), np.nan, dtype=np.float32) for _ in range(nrois)]

    # Per-distance, per-subject, per-pair arrays (like rssDistSess{iroi,idist})
    rssDistSess = [[np.full((nsubjects, nsessions - idist), np.nan, dtype=np.float32)
                    for idist in range(1, nsessions)] for _ in range(nrois)]
    rssOriDistSess = [[np.full((nsubjects, nsessions - idist), np.nan, dtype=np.float32)
                       for idist in range(1, nsessions)] for _ in range(nrois)]
    r2DistSess = [[np.full((nsubjects, nsessions - idist), np.nan, dtype=np.float32)
                   for idist in range(1, nsessions)] for _ in range(nrois)]
    r2OriDistSess = [[np.full((nsubjects, nsessions - idist), np.nan, dtype=np.float32)
                      for idist in range(1, nsessions)] for _ in range(nrois)]
    pearsonDistSess = [[np.full((nsubjects, nsessions - idist), np.nan, dtype=np.float32)
                        for idist in range(1, nsessions)] for _ in range(nrois)]
    pearsonOriDistSess = [[np.full((nsubjects, nsessions - idist), np.nan, dtype=np.float32)
                           for idist in range(1, nsessions)] for _ in range(nrois)]

    # Permutation versions of the above
    rssDistSessPerm = [[np.full((nsubjects, nperms, nsessions - idist), np.nan, dtype=np.float32)
                        for idist in range(1, nsessions)] for _ in range(nrois)]
    rssOriDistSessPerm = [[np.full((nsubjects, nperms, nsessions - idist), np.nan, dtype=np.float32)
                           for idist in range(1, nsessions)] for _ in range(nrois)]
    r2DistSessPerm = [[np.full((nsubjects, nperms, nsessions - idist), np.nan, dtype=np.float32)
                       for idist in range(1, nsessions)] for _ in range(nrois)]
    r2OriDistSessPerm = [[np.full((nsubjects, nperms, nsessions - idist), np.nan, dtype=np.float32)
                          for idist in range(1, nsessions)] for _ in range(nrois)]
    pearsonDistSessPerm = [[np.full((nsubjects, nperms, nsessions - idist), np.nan, dtype=np.float32)
                            for idist in range(1, nsessions)] for _ in range(nrois)]
    pearsonOriDistSessPerm = [[np.full((nsubjects, nperms, nsessions - idist), np.nan, dtype=np.float32)
                               for idist in range(1, nsessions)] for _ in range(nrois)]

    # Autocorrelation settings
    autocorrLength = 21  # lags 0..20
    acL = autocorrLength

    meanAutocorrBetas = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    autocorrMeanBetas = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    meanAutocorrStdBetas = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    autocorrMeanStdBetas = np.zeros((nrois, nsubjects, acL), dtype=np.float32)

    meanAutocorrConstant = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    meanAutocorrConstantOri = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    autocorrMeanConstant = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    autocorrMeanConstantOri = np.zeros((nrois, nsubjects, acL), dtype=np.float32)

    autocorrMeanCoef = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    autocorrMeanCoefOri = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    meanAutocorrCoef = np.zeros((nrois, nsubjects, acL), dtype=np.float32)
    meanAutocorrCoefOri = np.zeros((nrois, nsubjects, acL), dtype=np.float32)

    meanAutocorrBetasPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    meanAutocorrStdBetasPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    autocorrMeanBetasPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    autocorrMeanStdBetasPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    meanAutocorrConstantPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    meanAutocorrConstantOriPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    autocorrMeanConstantPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    autocorrMeanConstantOriPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    autocorrMeanCoefPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    autocorrMeanCoefOriPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    meanAutocorrCoefPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)
    meanAutocorrCoefOriPerm = np.zeros((nrois, nsubjects, nperms, acL), dtype=np.float32)

    # Per ROI x subject autocorr arrays (store per voxel)
    autocorrBetas = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrStdBetas = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrConstant = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrConstantOri = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrCoef = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrCoefOri = [[None for _ in range(nsubjects)] for _ in range(nrois)]

    autocorrBetasPerm = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrConstantPerm = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrConstantOriPerm = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrCoefPerm = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    autocorrCoefOriPerm = [[None for _ in range(nsubjects)] for _ in range(nrois)]

    # Other per-ROI×sub arrays
    subRoiPrf = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    numGoodVox = np.zeros((nrois, nsubjects), dtype=np.int32)
    totalVox = np.zeros((nrois, nsubjects), dtype=np.int32)
    r2GoodVox = np.zeros((nrois, nsubjects), dtype=np.float32)

    # This will hold std over coeffs (excluding constant) per session×voxel
    stdCoef = [[None for _ in range(nsubjects)] for _ in range(nrois)]
    stdOriCoef = [[None for _ in range(nsubjects)] for _ in range(nrois)]

    # permOrders per subject
    permOrders: Dict[int, np.ndarray] = {}



    # ----------------- Main loop over subjects ----------
    rng = np.random.default_rng(seed=1)  # similar to rng(1) in MATLAB

    for isub_idx, subj in enumerate(subjects):
        print(f"\nSubject {subj} (index {isub_idx})")
        in_name = f"regressSessCombineROI_sub{subj}{zscore_str}.pkl"
        in_path = os.path.join(save_dir, in_name)
        if not os.path.exists(in_path):
            print(f"  WARNING: missing {in_path}, skipping subject")
            continue

        with open(in_path, "rb") as f_in:
            data = pickle.load(f_in)

        allRoiPrf_list = data["allRoiPrf"]            # list length = numregions in that file
        roiNsdCorr_list = data["roiNsdCorr"]          # list of arrays
        roiNsdOriCorr_list = data["roiNsdOriCorr"]
        roiNsdOriR2_list = data["roiNsdOriR2"]
        roiNsdR2_list = data["roiNsdR2"]
        roiNsdOriRss_list = data["roiNsdOriRss"]
        roiNsdRss_list = data["roiNsdRss"]
        nsplits_combined = data["nsplits"]            # nsplits includes mean row
        sessBetas_list = data["sessBetas"]
        sessStdBetas_list = data["sessStdBetas"]
        voxConstCoef_list = data["voxConstCoef"]
        voxConstOriCoef_list = data["voxConstOriCoef"]
        voxOriCoef_list = data["voxOriCoef"]
        voxCoef_list = data["voxCoef"]

       # sessions in combined ROI file (exclude the last "mean" row)
        n_real = nsplits_combined - 1

        # how many sessions we will actually use for drift/perms
        use = min(nsessions, n_real)   # usually 30

        if use < 2:
            raise ValueError(f"Subject {subj}: not enough real sessions (n_real={n_real})")

        # Mark existing sessions (only those we use)
        subSessions[isub_idx, :use] = 1


        # ---------- Build permOrders for this subject ----------
        if isub_idx > 0:
            # use subject 1 permOrders for all others (as in MATLAB)
            permOrders[isub_idx] = permOrders[0]
        else:
            # create nperms permutations of 1..minSessions (here 1..30)
            po = np.zeros((nperms, minSessions), dtype=np.int32)
            for iperm in range(nperms):
                if fixedFirst:
                    tail = rng.permutation(np.arange(2, minSessions + 1))
                    po[iperm, :] = np.concatenate([np.array([1], dtype=np.int32), tail])
                else:
                    po[iperm, :] = rng.permutation(np.arange(1, minSessions + 1))
            permOrders[isub_idx] = po

        # ---------- Loop over requested ROIs ----------
        for iroi in rois:
            roi_idx = iroi - 1  # 0-based index into our arrays

            # take ROI iroi from the lists in the combined file
            roi_prf = allRoiPrf_list[roi_idx]  # dict, keys like 'ecc','ang','sz','x','y', maybe 'r2'
            roiNsdCorr = roiNsdCorr_list[roi_idx]      # (nsplits, nvox, nsplits)
            roiNsdOriCorr = roiNsdOriCorr_list[roi_idx]
            roiNsdOriR2 = roiNsdOriR2_list[roi_idx]
            roiNsdR2 = roiNsdR2_list[roi_idx]
            roiNsdOriRss = roiNsdOriRss_list[roi_idx]
            roiNsdRss = roiNsdRss_list[roi_idx]
            sessBetas_arr = sessBetas_list[roi_idx]    # (nsplits, nvox)
            sessStdBetas_arr = sessStdBetas_list[roi_idx]
            voxConstCoef_arr = voxConstCoef_list[roi_idx]      # (nsplits, nvox)
            voxConstOriCoef_arr = voxConstOriCoef_list[roi_idx]
            voxCoef_arr = voxCoef_list[roi_idx]                # (nsplits, nvox, P)
            voxOriCoef_arr = voxOriCoef_list[roi_idx]

            nvox = sessBetas_arr.shape[1]
            totalVox[roi_idx, isub_idx] = nvox

            # store pRF for plotting
            subRoiPrf[roi_idx][isub_idx] = roi_prf

            # good voxels by pRF r2 (if available) else all voxels
            if "r2" in roi_prf:
                prf_r2 = np.asarray(roi_prf["r2"]).ravel()
                goodVox_mask = prf_r2 > r2thresh
            else:
                goodVox_mask = np.ones(nvox, dtype=bool)

            r2GoodVox[roi_idx, isub_idx] = r2thresh
            numGoodVox[roi_idx, isub_idx] = int(goodVox_mask.sum())

            # --- session-mean metrics (over voxels) ---
            # use first nsessions rows (exclude mean row)
            sess_ix = np.arange(use)


            meanBetas[roi_idx, isub_idx, :use] = np.nanmean(sessBetas_arr[:use, goodVox_mask], axis=1)
            meanStdBetas[roi_idx, isub_idx, :use] = np.nanmean(sessStdBetas_arr[:use, goodVox_mask], axis=1)
            meanConstantCoef[roi_idx, isub_idx, :use] = np.nanmean(voxConstCoef_arr[:use, goodVox_mask], axis=1)
            meanConstantOriCoef[roi_idx, isub_idx, :use] = np.nanmean(voxConstOriCoef_arr[:use, goodVox_mask], axis=1)


            # std over coefficients excluding constant (last predictor)
            std_coef = np.std(voxCoef_arr[sess_ix, :, :-1], axis=2, ddof=0)      # (nsessions, nvox)
            std_ori_coef = np.std(voxOriCoef_arr[sess_ix, :, :-1], axis=2, ddof=0)

            stdCoef[roi_idx][isub_idx] = std_coef
            stdOriCoef[roi_idx][isub_idx] = std_ori_coef

            # Aggregated split×split matrices (median or mean across voxels)
            # roiNsdRss shape: (nsplits, nvox, nsplits)
            # We operate only on the real sessions: 0..use-1

            if useMedian:
                rssSplit_mat     = np.nanmedian(roiNsdRss[:use,     goodVox_mask, :use], axis=1)
                rssOriSplit_mat  = np.nanmedian(roiNsdOriRss[:use,  goodVox_mask, :use], axis=1)
                r2Split_mat      = np.nanmedian(roiNsdR2[:use,      goodVox_mask, :use], axis=1)
                r2OriSplit_mat   = np.nanmedian(roiNsdOriR2[:use,   goodVox_mask, :use], axis=1)
                pearsonR_mat     = np.nanmedian(roiNsdCorr[:use,    goodVox_mask, :use], axis=1)
                pearsonRori_mat  = np.nanmedian(roiNsdOriCorr[:use, goodVox_mask, :use], axis=1)
            else:
                rssSplit_mat     = np.nanmean(roiNsdRss[:use,     goodVox_mask, :use], axis=1)
                rssOriSplit_mat  = np.nanmean(roiNsdOriRss[:use,  goodVox_mask, :use], axis=1)
                r2Split_mat      = np.nanmean(roiNsdR2[:use,      goodVox_mask, :use], axis=1)
                r2OriSplit_mat   = np.nanmean(roiNsdOriR2[:use,   goodVox_mask, :use], axis=1)
                pearsonR_mat     = np.nanmean(roiNsdCorr[:use,    goodVox_mask, :use], axis=1)
                pearsonRori_mat  = np.nanmean(roiNsdOriCorr[:use, goodVox_mask, :use], axis=1)

            # Store only into valid session block (MATLAB-like behavior)
            rssSplit[roi_idx][isub_idx, :use, :use]     = rssSplit_mat
            rssOriSplit[roi_idx][isub_idx, :use, :use]  = rssOriSplit_mat
            r2split[roi_idx][isub_idx, :use, :use]      = r2Split_mat
            r2oriSplit[roi_idx][isub_idx, :use, :use]   = r2OriSplit_mat
            pearsonR[roi_idx][isub_idx, :use, :use]     = pearsonR_mat
            pearsonRori[roi_idx][isub_idx, :use, :use]  = pearsonRori_mat


            # ----- distance-based metrics (real order) -----
            for idist in range(1, nsessions):  # lags 1..29
                # 2D matrix of size nsessions × nsessions
                temp = rssSplit_mat[:minSessions, :minSessions]
                mask_pos = (lagMatrix == idist)
                mask_neg = (lagMatrix == -idist)
                v = 0.5 * (temp[mask_pos] + temp[mask_neg])   # length minSessions-idist
                rssDistSess[roi_idx][idist - 1][isub_idx, :] = v
                rssDist[roi_idx][isub_idx, idist - 1] = np.nanmean(temp[distMatrix == idist])

                temp = rssOriSplit_mat[:minSessions, :minSessions]
                v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                rssOriDistSess[roi_idx][idist - 1][isub_idx, :] = v
                rssOriDist[roi_idx][isub_idx, idist - 1] = np.nanmean(temp[distMatrix == idist])

                temp = r2Split_mat[:minSessions, :minSessions]
                v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                r2DistSess[roi_idx][idist - 1][isub_idx, :] = v
                r2Dist[roi_idx][isub_idx, idist - 1] = np.nanmean(temp[distMatrix == idist])

                temp = r2OriSplit_mat[:minSessions, :minSessions]
                v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                r2OriDistSess[roi_idx][idist - 1][isub_idx, :] = v
                r2OriDist[roi_idx][isub_idx, idist - 1] = np.nanmean(temp[distMatrix == idist])

                temp = pearsonR_mat[:minSessions, :minSessions]
                v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                pearsonDistSess[roi_idx][idist - 1][isub_idx, :] = v
                pearsonDist[roi_idx][isub_idx, idist - 1] = np.nanmean(temp[distMatrix == idist])

                temp = pearsonRori_mat[:minSessions, :minSessions]
                v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                pearsonOriDistSess[roi_idx][idist - 1][isub_idx, :] = v
                pearsonOriDist[roi_idx][isub_idx, idist - 1] = np.nanmean(temp[distMatrix == idist])

            # ----- Autocorrelations (true order; EXCLUDE mean row) -----
            n_real = nsplits_combined - 1          # real sessions in this file (drops mean row)

            nvox_roi = nvox
            ac_betas = np.zeros((nvox_roi, acL), dtype=np.float32)
            ac_std_betas = np.zeros((nvox_roi, acL), dtype=np.float32)
            ac_const = np.zeros((nvox_roi, acL), dtype=np.float32)
            ac_const_ori = np.zeros((nvox_roi, acL), dtype=np.float32)
            ac_coef = np.zeros((nvox_roi, acL), dtype=np.float32)
            ac_coef_ori = np.zeros((nvox_roi, acL), dtype=np.float32)

            # Make sure these are session-aligned and exclude mean row too
            sessBetas_ts       = sessBetas_arr[:use, :]           # (use, nvox)
            sessStdBetas_ts    = sessStdBetas_arr[:use, :]
            voxConstCoef_ts    = voxConstCoef_arr[:use, :]
            voxConstOriCoef_ts = voxConstOriCoef_arr[:use, :]
            std_coef_ts        = std_coef[:use, :]                # std_coef was computed from sess_ix already, but slice anyway
            std_ori_coef_ts    = std_ori_coef[:use, :]

            for ivox in range(nvox_roi):
                ac_betas[ivox, :] = autocorr_1d(sessBetas_ts[:, ivox], acL - 1)
                ac_std_betas[ivox, :] = autocorr_1d(sessStdBetas_ts[:, ivox], acL - 1)
                ac_const[ivox, :] = autocorr_1d(voxConstCoef_ts[:, ivox], acL - 1)
                ac_const_ori[ivox, :] = autocorr_1d(voxConstOriCoef_ts[:, ivox], acL - 1)
                ac_coef[ivox, :] = autocorr_1d(std_coef_ts[:, ivox], acL - 1)
                ac_coef_ori[ivox, :] = autocorr_1d(std_ori_coef_ts[:, ivox], acL - 1)

            autocorrBetas[roi_idx][isub_idx] = ac_betas
            autocorrStdBetas[roi_idx][isub_idx] = ac_std_betas
            autocorrConstant[roi_idx][isub_idx] = ac_const
            autocorrConstantOri[roi_idx][isub_idx] = ac_const_ori
            autocorrCoef[roi_idx][isub_idx] = ac_coef
            autocorrCoefOri[roi_idx][isub_idx] = ac_coef_ori


            # mean across good voxels vs autocorr of mean across voxels
            good_ix = np.where(goodVox_mask)[0]
            meanAutocorrBetas[roi_idx, isub_idx, :] = np.nanmean(ac_betas[good_ix, :], axis=0)
            autocorrMeanBetas[roi_idx, isub_idx, :] = autocorr_1d(meanBetas[roi_idx, isub_idx, :], acL - 1)
            meanAutocorrStdBetas[roi_idx, isub_idx, :] = np.nanmean(ac_std_betas[good_ix, :], axis=0)
            autocorrMeanStdBetas[roi_idx, isub_idx, :] = autocorr_1d(meanStdBetas[roi_idx, isub_idx, :], acL - 1)

            meanAutocorrConstant[roi_idx, isub_idx, :] = np.nanmean(ac_const[good_ix, :], axis=0)
            meanAutocorrConstantOri[roi_idx, isub_idx, :] = np.nanmean(ac_const_ori[good_ix, :], axis=0)
            autocorrMeanConstant[roi_idx, isub_idx, :] = autocorr_1d(meanConstantCoef[roi_idx, isub_idx, :], acL - 1)
            autocorrMeanConstantOri[roi_idx, isub_idx, :] = autocorr_1d(meanConstantOriCoef[roi_idx, isub_idx, :], acL - 1)

            autocorrMeanCoef[roi_idx, isub_idx, :] = autocorr_1d(np.nanmean(std_coef[:, good_ix], axis=1), acL - 1)
            autocorrMeanCoefOri[roi_idx, isub_idx, :] = autocorr_1d(np.nanmean(std_ori_coef[:, good_ix], axis=1), acL - 1)
            meanAutocorrCoef[roi_idx, isub_idx, :] = np.nanmean(ac_coef[good_ix, :], axis=0)
            meanAutocorrCoefOri[roi_idx, isub_idx, :] = np.nanmean(ac_coef_ori[good_ix, :], axis=0)

            # Allocate perm autocorr arrays for this ROI×subject
            autocorrBetasPerm[roi_idx][isub_idx] = np.zeros((nperms, nvox_roi, acL), dtype=np.float32)
            autocorrConstantPerm[roi_idx][isub_idx] = np.zeros((nperms, nvox_roi, acL), dtype=np.float32)
            autocorrConstantOriPerm[roi_idx][isub_idx] = np.zeros((nperms, nvox_roi, acL), dtype=np.float32)
            autocorrCoefPerm[roi_idx][isub_idx] = np.zeros((nperms, nvox_roi, acL), dtype=np.float32)
            autocorrCoefOriPerm[roi_idx][isub_idx] = np.zeros((nperms, nvox_roi, acL), dtype=np.float32)

            # ---------- Permutations for this ROI×subject ----------
            po_sub = permOrders[isub_idx]  # (nperms, minSessions)

            for iperm in range(nperms):
                permOrder = po_sub[iperm, :] - 1  # convert to 0-based indices

                # permute split×split matrices (30×30 subset)
                temp_rss = rssSplit_mat[permOrder[:, None], permOrder[None, :]]
                temp_rss_ori = rssOriSplit_mat[permOrder[:, None], permOrder[None, :]]
                temp_r2 = r2Split_mat[permOrder[:, None], permOrder[None, :]]
                temp_r2_ori = r2OriSplit_mat[permOrder[:, None], permOrder[None, :]]
                temp_p = pearsonR_mat[permOrder[:, None], permOrder[None, :]]
                temp_p_ori = pearsonRori_mat[permOrder[:, None], permOrder[None, :]]

                rssPerm[roi_idx][isub_idx, iperm, :, :] = temp_rss
                rssOriPerm[roi_idx][isub_idx, iperm, :, :] = temp_rss_ori
                r2perm[roi_idx][isub_idx, iperm, :, :] = temp_r2
                r2oriPerm[roi_idx][isub_idx, iperm, :, :] = temp_r2_ori
                pearsonPerm[roi_idx][isub_idx, iperm, :, :] = temp_p
                pearsonOriPerm[roi_idx][isub_idx, iperm, :, :] = temp_p_ori

                # distance-based metrics for permuted order
                for idist in range(1, nsessions):
                    mask_pos = (lagMatrix == idist)
                    mask_neg = (lagMatrix == -idist)

                    temp = temp_rss
                    v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                    rssDistSessPerm[roi_idx][idist - 1][isub_idx, iperm, :] = v
                    rssDistPerm[roi_idx][isub_idx, iperm, idist - 1] = np.nanmean(temp[distMatrix == idist])

                    temp = temp_rss_ori
                    v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                    rssOriDistSessPerm[roi_idx][idist - 1][isub_idx, iperm, :] = v
                    rssOriDistPerm[roi_idx][isub_idx, iperm, idist - 1] = np.nanmean(temp[distMatrix == idist])

                    temp = temp_r2
                    v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                    r2DistSessPerm[roi_idx][idist - 1][isub_idx, iperm, :] = v
                    r2DistPerm[roi_idx][isub_idx, iperm, idist - 1] = np.nanmean(temp[distMatrix == idist])

                    temp = temp_r2_ori
                    v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                    r2OriDistSessPerm[roi_idx][idist - 1][isub_idx, iperm, :] = v
                    r2OriDistPerm[roi_idx][isub_idx, iperm, idist - 1] = np.nanmean(temp[distMatrix == idist])

                    temp = temp_p
                    v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                    pearsonDistSessPerm[roi_idx][idist - 1][isub_idx, iperm, :] = v
                    pearsonDistPerm[roi_idx][isub_idx, iperm, idist - 1] = np.nanmean(temp[distMatrix == idist])

                    temp = temp_p_ori
                    v = 0.5 * (temp[mask_pos] + temp[mask_neg])
                    pearsonOriDistSessPerm[roi_idx][idist - 1][isub_idx, iperm, :] = v
                    pearsonOriDistPerm[roi_idx][isub_idx, iperm, idist - 1] = np.nanmean(temp[distMatrix == idist])

                # Autocorrelations for permuted order
                # permute sessions for each voxel
                # use the same "true-order" sliced time series (length = use)
                sess_only = sessBetas_arr[:use, :]
                std_only  = sessStdBetas_arr[:use, :]
                c_only    = voxConstCoef_arr[:use, :]
                co_only   = voxConstOriCoef_arr[:use, :]
                std_coef_only     = std_coef[:use, :]
                std_ori_coef_only = std_ori_coef[:use, :]

                sess_perm       = sess_only[permOrder, :]
                sess_std_perm   = std_only[permOrder, :]
                const_perm      = c_only[permOrder, :]
                const_ori_perm  = co_only[permOrder, :]
                std_coef_perm   = std_coef_only[permOrder, :]
                std_ori_coef_perm = std_ori_coef_only[permOrder, :]



                ac_betas_perm = np.zeros((nvox_roi, acL), dtype=np.float32)
                ac_std_perm = np.zeros((nvox_roi, acL), dtype=np.float32)
                ac_const_perm = np.zeros((nvox_roi, acL), dtype=np.float32)
                ac_const_ori_perm = np.zeros((nvox_roi, acL), dtype=np.float32)
                ac_coef_perm = np.zeros((nvox_roi, acL), dtype=np.float32)
                ac_coef_ori_perm = np.zeros((nvox_roi, acL), dtype=np.float32)

                for ivox in range(nvox_roi):
                    ac_betas_perm[ivox, :] = autocorr_1d(sess_perm[:, ivox], acL - 1)
                    ac_std_perm[ivox, :] = autocorr_1d(sess_std_perm[:, ivox], acL - 1)
                    ac_const_perm[ivox, :] = autocorr_1d(const_perm[:, ivox], acL - 1)
                    ac_const_ori_perm[ivox, :] = autocorr_1d(const_ori_perm[:, ivox], acL - 1)
                    ac_coef_perm[ivox, :] = autocorr_1d(std_coef_perm[:, ivox], acL - 1)
                    ac_coef_ori_perm[ivox, :] = autocorr_1d(std_ori_coef_perm[:, ivox], acL - 1)

                autocorrBetasPerm[roi_idx][isub_idx][iperm, :, :] = ac_betas_perm
                autocorrConstantPerm[roi_idx][isub_idx][iperm, :, :] = ac_const_perm
                autocorrConstantOriPerm[roi_idx][isub_idx][iperm, :, :] = ac_const_ori_perm
                autocorrCoefPerm[roi_idx][isub_idx][iperm, :, :] = ac_coef_perm
                autocorrCoefOriPerm[roi_idx][isub_idx][iperm, :, :] = ac_coef_ori_perm

                # mean across voxels / good voxels
                meanAutocorrBetasPerm[roi_idx, isub_idx, iperm, :] = np.nanmean(ac_betas_perm[good_ix, :], axis=0)
                meanAutocorrStdBetasPerm[roi_idx, isub_idx, iperm, :] = np.nanmean(ac_std_perm[good_ix, :], axis=0)
                autocorrMeanBetasPerm[roi_idx, isub_idx, iperm, :] = autocorr_1d(meanBetas[roi_idx, isub_idx, permOrder], acL - 1)
                autocorrMeanStdBetasPerm[roi_idx, isub_idx, iperm, :] = autocorr_1d(meanStdBetas[roi_idx, isub_idx, permOrder], acL - 1)

                meanAutocorrConstantPerm[roi_idx, isub_idx, iperm, :] = np.nanmean(ac_const_perm[good_ix, :], axis=0)
                meanAutocorrConstantOriPerm[roi_idx, isub_idx, iperm, :] = np.nanmean(ac_const_ori_perm[good_ix, :], axis=0)
                autocorrMeanConstantPerm[roi_idx, isub_idx, iperm, :] = autocorr_1d(meanConstantCoef[roi_idx, isub_idx, permOrder], acL - 1)
                autocorrMeanConstantOriPerm[roi_idx, isub_idx, iperm, :] = autocorr_1d(meanConstantOriCoef[roi_idx, isub_idx, permOrder], acL - 1)

                autocorrMeanCoefPerm[roi_idx, isub_idx, iperm, :] = autocorr_1d(np.nanmean(std_coef_perm[:, good_ix], axis=1), acL - 1)
                autocorrMeanCoefOriPerm[roi_idx, isub_idx, iperm, :] = autocorr_1d(np.nanmean(std_ori_coef_perm[:, good_ix], axis=1), acL - 1)
                meanAutocorrCoefPerm[roi_idx, isub_idx, iperm, :] = np.nanmean(ac_coef_perm[good_ix, :], axis=0)
                meanAutocorrCoefOriPerm[roi_idx, isub_idx, iperm, :] = np.nanmean(ac_coef_ori_perm[good_ix, :], axis=0)

    # ----------------- Save -----------------
    normalizeStr = "_normalized" if toNormalize else ""

    rois_str = "roi" + "_".join(map(str, rois))
    out_name = f"perm_{rois_str}{fixedFirstStr}{nperms}{zscore_str}{normalizeStr}{r2threshStr}.pkl"
    out_path = os.path.join(save_dir, out_name)

    out_dict = {
        "toNormalize": toNormalize,
        "toZscore": to_zscore,
        "useMedian": useMedian,
        "r2thresh": r2thresh,
        "nrois": nrois,
        "rois": rois,
        "permOrders": permOrders,
        "subSessions": subSessions,
        "subjects": subjects,
        "minSessions": minSessions,
        "r2split": r2split,
        "r2oriSplit": r2oriSplit,
        "pearsonRori": pearsonRori,
        "pearsonR": pearsonR,
        "r2Dist": r2Dist,
        "r2OriDist": r2OriDist,
        "pearsonDist": pearsonDist,
        "pearsonOriDist": pearsonOriDist,
        "r2DistPerm": r2DistPerm,
        "r2OriDistPerm": r2OriDistPerm,
        "pearsonDistPerm": pearsonDistPerm,
        "pearsonOriDistPerm": pearsonOriDistPerm,
        "r2DistSess": r2DistSess,
        "r2OriDistSess": r2OriDistSess,
        "pearsonDistSess": pearsonDistSess,
        "pearsonOriDistSess": pearsonOriDistSess,
        "rssDistSess": rssDistSess,
        "rssOriDistSess": rssOriDistSess,
        "rssDist": rssDist,
        "rssOriDist": rssOriDist,
        "rssSplit": rssSplit,
        "rssOriSplit": rssOriSplit,
        "rssDistPerm": rssDistPerm,
        "rssOriDistPerm": rssOriDistPerm,
        "rssDistSessPerm": rssDistSessPerm,
        "rssOriDistSessPerm": rssOriDistSessPerm,
        "r2DistSessPerm": r2DistSessPerm,
        "r2OriDistSessPerm": r2OriDistSessPerm,
        "pearsonDistSessPerm": pearsonDistSessPerm,
        "pearsonOriDistSessPerm": pearsonOriDistSessPerm,
        "meanAutocorrBetas": meanAutocorrBetas,
        "autocorrMeanBetas": autocorrMeanBetas,
        "meanAutocorrStdBetas": meanAutocorrStdBetas,
        "autocorrMeanStdBetas": autocorrMeanStdBetas,
        "meanAutocorrConstant": meanAutocorrConstant,
        "meanAutocorrConstantOri": meanAutocorrConstantOri,
        "autocorrMeanConstant": autocorrMeanConstant,
        "autocorrMeanConstantOri": autocorrMeanConstantOri,
        "autocorrMeanCoef": autocorrMeanCoef,
        "autocorrMeanCoefOri": autocorrMeanCoefOri,
        "meanAutocorrCoef": meanAutocorrCoef,
        "meanAutocorrCoefOri": meanAutocorrCoefOri,
        "meanAutocorrBetasPerm": meanAutocorrBetasPerm,
        "autocorrMeanBetasPerm": autocorrMeanBetasPerm,
        "meanBetas": meanBetas,
        "meanAutocorrStdBetasPerm": meanAutocorrStdBetasPerm,
        "autocorrMeanStdBetasPerm": autocorrMeanStdBetasPerm,
        "meanAutocorrConstantPerm": meanAutocorrConstantPerm,
        "meanAutocorrConstantOriPerm": meanAutocorrConstantOriPerm,
        "autocorrMeanConstantPerm": autocorrMeanConstantPerm,
        "autocorrMeanConstantOriPerm": autocorrMeanConstantOriPerm,
        "autocorrMeanCoefPerm": autocorrMeanCoefPerm,
        "autocorrMeanCoefOriPerm": autocorrMeanCoefOriPerm,
        "meanAutocorrCoefPerm": meanAutocorrCoefPerm,
        "meanAutocorrCoefOriPerm": meanAutocorrCoefOriPerm,
        "subRoiPrf": subRoiPrf,
        "numGoodVox": numGoodVox,
        "totalVox": totalVox,
        "r2perm": r2perm,
        "r2oriPerm": r2oriPerm,
        "pearsonPerm": pearsonPerm,
        "pearsonOriPerm": pearsonOriPerm,
        "rssPerm": rssPerm,
        "rssOriPerm": rssOriPerm,
    }

    with open(out_path, "wb") as f_out:
        pickle.dump(out_dict, f_out, protocol=pickle.HIGHEST_PROTOCOL)

    dt = time.time() - t0
    print(f"\nSaved permutation results to: {out_path}")
    print(f"Total time: {dt/60:.1f} minutes (approx)")

    return out_path


# ============================================================
# RUN PERMUTATION ANALYSIS
# Example:

#     rois=[3]        -> run V3
#     to_zscore=2    -> use zeroMean files
#     nperms=1000    -> generate 1000 shuffled session orders
#     subjects=[1..8] -> include all NSD subjects
#
# For quick testing, use nperms=10 before running the full analysis.
# ============================================================
out_file = save_perms_expand(rois=[3], to_zscore=2, nperms=1000, r2thresh=0.0, fixedFirst=False, subjects=[1, 2 ,3, 4, 5, 6, 7, 8])
print("Output file:", out_file)


# CHECK SAVED PERMUTATION FILE
 This diagnostic cell checks that the permutation output file:
#
     1. exists
     2. can be loaded
     3. contains the expected MATLAB-compatible variables
     4. has reasonable shapes and numerical values
#
 Run this after save_perms_expand finishes.


In [ ]:
# Inspect a save_perms_expand output .pkl (Colab-safe + works outside Colab too)

import os
import pickle
import numpy as np

# ---- Optional: mount Drive only if we're in Colab ----
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)

# ---- Path to your output ----
out_path = "/content/drive/MyDrive/V1_Drift/repDrift_expand/perm_roi11000.pkl"
assert os.path.exists(out_path), f"File not found: {out_path}"

print("Found:", out_path)
print("Size (GB):", os.path.getsize(out_path) / (1024**3))

# ---- Load ----
with open(out_path, "rb") as f:
    d = pickle.load(f)

print("\n=== Top-level keys ===")
keys = sorted(d.keys())
print(f"n_keys = {len(keys)}")
for k in keys:
    print(" -", k)

# ---- Helpers ----
def finite_stats(arr: np.ndarray):
    arr = np.asarray(arr)
    fin = np.isfinite(arr)
    n = arr.size
    nfin = int(fin.sum())
    out = {
        "n": n,
        "finite": nfin,
        "nan%": float(np.isnan(arr).mean() * 100) if n else 0.0,
        "inf%": float(np.isinf(arr).mean() * 100) if n else 0.0,
    }
    if nfin:
        x = arr[fin]
        out.update(
            mean=float(x.mean()),
            std=float(x.std()),
            min=float(x.min()),
            max=float(x.max()),
        )
    return out

def summarize(name, obj, max_list_peek=2):
    if isinstance(obj, np.ndarray):
        print(f"{name:30s} ndarray  shape={obj.shape} dtype={obj.dtype}")
        if np.issubdtype(obj.dtype, np.number) and obj.size:
            st = finite_stats(obj)
            print(
                f"{'':30s} stats   fin={st['finite']}/{st['n']}  "
                f"NaN={st['nan%']:.3f}%  Inf={st['inf%']:.3f}%  "
                f"mean={st.get('mean', np.nan):.5g}  std={st.get('std', np.nan):.5g}  "
                f"min={st.get('min', np.nan):.5g}  max={st.get('max', np.nan):.5g}"
            )
        return

    if isinstance(obj, list):
        print(f"{name:30s} list    len={len(obj)}")
        for i, el in enumerate(obj[:max_list_peek]):
            if isinstance(el, np.ndarray):
                print(f"{'':30s} [{i}] ndarray shape={el.shape} dtype={el.dtype}")
            elif isinstance(el, list):
                print(f"{'':30s} [{i}] list len={len(el)}")
                if len(el) and isinstance(el[0], np.ndarray):
                    print(f"{'':30s}     [0] ndarray shape={el[0].shape} dtype={el[0].dtype}")
            else:
                print(f"{'':30s} [{i}] type={type(el).__name__}")
        return

    if isinstance(obj, dict):
        ks = list(obj.keys())
        print(f"{name:30s} dict    len={len(obj)} keys(sample)={ks[:5]}")
        if len(ks):
            k0 = ks[0]
            v0 = obj[k0]
            if isinstance(v0, np.ndarray):
                print(f"{'':30s} [{k0}] ndarray shape={v0.shape} dtype={v0.dtype}")
            else:
                print(f"{'':30s} [{k0}] type={type(v0).__name__}")
        return

    print(f"{name:30s} {type(obj).__name__}: {str(obj)[:120]}")

# ---- Check “MATLAB saved variables” coverage ----
matlab_saved = [
    "toNormalize","toZscore","useMedian","r2thresh","nrois","rois",
    "permOrders","subSessions","subjects","minSessions",
    "r2split","r2oriSplit","pearsonRori","pearsonR",
    "r2Dist","r2OriDist","pearsonDist","pearsonOriDist",
    "r2DistPerm","r2OriDistPerm","pearsonDistPerm","pearsonOriDistPerm",
    "r2DistSess","r2OriDistSess","pearsonDistSess","pearsonOriDistSess",
    "rssDistSess","rssOriDistSess",
    "rssDist","rssOriDist",
    "rssSplit","rssOriSplit",
    "rssDistPerm","rssOriDistPerm",
    "rssDistSessPerm","rssOriDistSessPerm",
    "r2DistSessPerm","r2OriDistSessPerm","pearsonDistSessPerm","pearsonOriDistSessPerm",
    "meanAutocorrBetas","autocorrMeanBetas","meanAutocorrStdBetas","autocorrMeanStdBetas",
    "meanAutocorrConstant","meanAutocorrConstantOri","autocorrMeanConstant","autocorrMeanConstantOri",
    "autocorrMeanCoef","autocorrMeanCoefOri","meanAutocorrCoef","meanAutocorrCoefOri",
    "meanAutocorrBetasPerm","autocorrMeanBetasPerm","meanBetas",
    "meanAutocorrStdBetasPerm","autocorrMeanStdBetasPerm",
    "meanAutocorrConstantPerm","meanAutocorrConstantOriPerm",
    "autocorrMeanConstantPerm","autocorrMeanConstantOriPerm",
    "autocorrMeanCoefPerm","autocorrMeanCoefOriPerm",
    "meanAutocorrCoefPerm","meanAutocorrCoefOriPerm",
    "subRoiPrf","numGoodVox","totalVox",
    "r2perm","r2oriPerm","pearsonPerm","pearsonOriPerm",
    "rssPerm","rssOriPerm",
]

missing = [k for k in matlab_saved if k not in d]
present = [k for k in matlab_saved if k in d]

print("\n=== MATLAB-variable coverage ===")
print(f"Present: {len(present)}/{len(matlab_saved)}")
if missing:
    print("Missing keys:")
    for k in missing:
        print(" -", k)

# ---- Summarize key structures ----
print("\n=== Key summaries ===")
for k in [
    "toNormalize","toZscore","useMedian","r2thresh","rois","subjects","minSessions",
    "subSessions","permOrders",
    "rssSplit","r2split","pearsonR",
    "rssDist","r2Dist","pearsonDist",
    "rssDistPerm","r2DistPerm","pearsonDistPerm",
    "rssPerm","r2perm","pearsonPerm",
    "meanBetas",
    "meanAutocorrBetas","autocorrMeanBetas",
    "meanAutocorrBetasPerm","autocorrMeanBetasPerm",
]:
    if k in d:
        summarize(k, d[k])

# ---- Sanity checks ----
print("\n=== Sanity checks ===")
# permOrders should be (nperms, 30) for subject-0/subject-1 depending on your indexing
po = d.get("permOrders", None)
if isinstance(po, dict) and len(po):
    k0 = sorted(po.keys())[0]
    arr = po[k0]
    print(f"permOrders[{k0}] shape={arr.shape} dtype={arr.dtype} min={arr.min()} max={arr.max()}")
elif isinstance(po, list) and len(po):
    arr = np.asarray(po[0])
    print(f"permOrders[0] shape={arr.shape} dtype={arr.dtype} min={arr.min()} max={arr.max()}")
else:
    print("permOrders type:", type(po).__name__)

ss = np.asarray(d.get("subSessions"))
print("subSessions shape:", ss.shape, "sum:", int(ss.sum()))

# Example: check ROI-1 split matrix for subject 0 (if lists-of-arrays)
rssSplit = d.get("rssSplit", None)
if isinstance(rssSplit, list) and len(rssSplit) >= 1:
    x = rssSplit[0]  # ROI 1
    print("rssSplit[ROI1] shape:", np.asarray(x).shape, "dtype:", np.asarray(x).dtype)
    # subject 0, lags diag stats
    x0 = np.asarray(x)[0]
    diag = np.diag(x0)
    print("ROI1 subj0 diag stats:", finite_stats(diag))
